COMP 3608 Project

Imports

In [ ]:
import pandas as pd
import numpy as np

Reading Data and initial data exploration

In [ ]:
df_A_2024 = pd.read_csv("./data/vgchartz-2024.csv")
df_B_2024 = pd.read_csv("./data/Video_Games.csv")
df_C_2026 = pd.read_csv("./data/vgsales.csv")

In [ ]:
df_A_2024.head()

In [ ]:
df_B_2024.head()

In [ ]:
df_C_2026.head()

Unneccesary Column Removal

In [ ]:
df_A_rmC = df_A_2024.drop(columns=['img','developer','critic_score','total_sales','last_update'])
df_A_rmC.head()

In [ ]:
df_B_rmC = df_B_2024.drop(columns=['index','Global_Sales','Critic_Score','Critic_Count','User_Score','User_Count','Developer','Rating'])
df_B_rmC.head()

In [ ]:
df_C_rmC = df_C_2026.drop(columns=['Rank','Global_Sales'])
df_C_rmC.head()

Renaming and Reordering

In [ ]:
A_rename_mapping = {'title':'Name'
                    ,'console':'Platform'
                    ,'genre':'Genre'
                    ,'publisher':'Publisher'
                    ,'na_sales':'NA_Sales'
                    ,'jp_sales':'JP_Sales'
                    ,'pal_sales':'EU_Sales'
                    ,'other_sales':'Other_Sales'
                    ,'release_date':'Release_Year'}

df_A_rename = df_A_rmC.rename(columns=A_rename_mapping)
df_A_rename.head()

In [ ]:
B_rename_mapping = {'Year_of_Release':'Release_Year'}

df_B_rename = df_B_rmC.rename(columns=B_rename_mapping)
df_B_rename.head()

In [ ]:
C_rename_mapping = {'Year':'Release_Year'}

df_C_rename = df_C_rmC.rename(columns=C_rename_mapping)
df_C_rename.head()

In [ ]:
new_order = ['Name','Publisher','Platform','Genre','Release_Year','NA_Sales','EU_Sales','JP_Sales','Other_Sales']

df_A_reorder = df_A_rename[new_order]
df_A_reorder.head()

In [ ]:
df_B_reorder = df_B_rename[new_order]
df_B_reorder.head()

In [ ]:
df_C_reorder = df_C_rename[new_order]
df_C_reorder.head()

Removing Duplicate/Blank Data

In [ ]:
df_A_sorted = df_A_reorder.sort_values(by='Name',ascending=True)
df_B_sorted = df_B_reorder.sort_values(by='Name',ascending=True)
df_C_sorted = df_C_reorder.sort_values(by='Name',ascending=True)

In [ ]:
df_A_sorted.head()

In [ ]:
df_B_sorted.head()

In [ ]:
df_C_sorted.head()

In [ ]:
df_A_removeNull = df_A_sorted.dropna().copy()
df_B_removeNull = df_B_sorted.dropna().copy()
df_C_removeNull = df_C_sorted.dropna().copy()

In [ ]:
df_A_removeNull.head()

In [ ]:
df_B_removeNull.head()

In [ ]:
df_C_removeNull.head()

In [ ]:
df_A_removeNull['_source'] = 'vgchartz'
df_B_removeNull['_source'] = 'videogames'

In [ ]:
df_AB_merged = pd.concat([df_A_removeNull, df_B_removeNull], ignore_index=True)
df_AB_merged = df_AB_merged.drop_duplicates()

In [ ]:
df_AB_merged.head()

In [ ]:
query_columns = ['Name','Publisher','Platform']

df_C_filtered = df_C_removeNull.merge(df_AB_merged[query_columns], how='inner', on=query_columns)
df_C_filtered = df_C_filtered.drop_duplicates()

In [ ]:
df_C_filtered.head()

De-duplication

In [ ]:
df_AB_merged['key'] = df_AB_merged['Name'] + '|' + df_AB_merged['Publisher'] + '|' + df_AB_merged['Platform']
df_C_filtered['key'] = df_C_filtered['Name'] + '|' + df_C_filtered['Publisher'] + '|' + df_C_filtered['Platform']

common_keys = set(df_AB_merged['key']) & set(df_C_filtered['key'])

df_A_intersect = df_AB_merged[df_AB_merged['key'].isin(common_keys)].copy()
df_C_intersect = df_C_filtered[df_C_filtered['key'].isin(common_keys)].copy()

df_A_intersect['_source_priority'] = df_A_intersect['_source'].map({'videogames': 0, 'vgchartz': 1})
df_A_intersect = (df_A_intersect
                  .sort_values(['key', '_source_priority'])
                  .drop_duplicates(subset=['key'], keep='first')
                  .drop(columns=['key', '_source', '_source_priority']))

df_C_intersect = (df_C_intersect
                  .drop_duplicates(subset=['key'], keep='first')
                  .drop(columns=['key']))

df_A_intersect = df_A_intersect.sort_values(by=['Name', 'Publisher', 'Platform']).reset_index(drop=True)
df_C_intersect = df_C_intersect.sort_values(by=['Name', 'Publisher', 'Platform']).reset_index(drop=True)

In [ ]:
df_A_intersect.head()

In [ ]:
df_C_intersect.head()

Exporting final dataframes to new files

In [ ]:
df_A_intersect.to_csv('data/vgData2024.csv', index=False)

In [ ]:
df_C_intersect.to_csv('data/vgData2026.csv', index=False)